# How do Fed policy changes affect market volatility?

**FinTechCo Data Science · Exploratory analysis on public FRED data**

> *"How do Federal Reserve policy changes historically impact market volatility, and can we quantify these relationships?"*

This notebook answers the question in four steps, each with a chart and a table you can extend:

| # | Approach | What it tells us |
|---|---|---|
| 1 | **Context**: policy rate and VIX since 1990 | When the Fed moves, and how that lines up with volatility |
| 2 | **Event study with placebo test** | Do rate *decisions* move VIX more than an ordinary day at the same VIX level? |
| 3 | **Policy regimes** | Is volatility different in tightening, easing, and on-hold periods? |
| 4 | **Surprise regressions (local projections)** | Does the *unexpected* part of a decision move VIX, by how much, and for how long? |

## Key findings

*Numbers as of the FRED pull on 2026-09-25 · 92 FOMC rate changes since Feb 1994 · VIX = 30-day implied S&P 500 volatility.*

1. **The Fed cuts into stress and hikes into calm.** VIX averages **27.2 the day before a cut** vs. **17.1 before a hike**, and 53% of cuts happen with VIX ≥ 25. Causality runs strongly from volatility *to* policy, and every result below has to account for that.
2. **Rate decisions by themselves don't measurably move VIX.** VIX falls about 1.4 points in the 20 days after a cut. But random days at the *same* VIX level fall by about 1.8 points, so that's mean reversion, not the Fed (p = 0.64). No horizon, for hikes or cuts, beats the placebo (all p ≥ 0.14).
3. **Volatility is higher in easing regimes, but that's association, not effect.** Average VIX is **22.8 in easing periods vs. 16.6 in tightening** (realized NASDAQ volatility 24.6% vs. 18.4%). Given finding 1, read this as "easing cycles start in turbulent markets."
4. **Only surprises matter, only in stress, and only briefly.** A +25 bp hawkish surprise (2-year yield) is associated with **−2.2 VIX points** on the decision day (p = 0.04). The effect comes entirely from stress-period decisions (**−4.0 pts**, p < 0.01). For routine decisions it is **zero** (+0.06, p = 0.92), and it is gone by the next day. In a crisis, an unexpectedly *dovish* move raises VIX: markets read it as the Fed seeing more trouble than they did (the "Fed information effect").

**So what for portfolio strategy:** routine, well-communicated FOMC decisions are not a volatility event. Policy-related volatility risk is concentrated in **stress periods with surprise easing**, so position around market stress and the surprise, not around the decision date.

## 0 · Setup

All I/O goes through `fintechco_demo.fred_client`. It reads `FRED_API_KEY` from `.env` (never hardcoded) and caches raw pulls in `data/raw/`, so the notebook reruns offline and every number traces back to a dated file.

In [1]:
import logging

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from fintechco_demo import analysis as an
from fintechco_demo import charts
from fintechco_demo import cleaning as cl
from fintechco_demo import fred_client as fc

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)
pd.set_option("display.float_format", "{:,.2f}".format)

EVENT_SAMPLE_START = "1994-02-04"  # first FOMC decision announced by press release
STRESS_VIX = 25  # VIX level the day before that marks a "stress-period" decision

## 1 · Data acquisition

| FRED ID | What it is | Role in the analysis |
|---|---|---|
| `VIXCLS` | CBOE VIX: 30-day implied volatility of S&P 500 options | **Outcome**: forward-looking market volatility |
| `NASDAQCOM` | NASDAQ Composite index | Realized volatility (robustness check) |
| `DFEDTAR` | Fed funds target rate (until 2008-12-15) | Policy decisions |
| `DFEDTARU` | Fed funds target range, upper bound (from 2008-12-16) | Policy decisions |
| `DGS2` | 2-year Treasury yield | **Surprise proxy**: the decision-day move captures news about the policy path the market had *not* priced in |

In [2]:
SERIES = ["VIXCLS", "NASDAQCOM", "DFEDTAR", "DFEDTARU", "DGS2"]
raw = fc.fetch_many(SERIES)

pd.DataFrame(
    {
        sid: {
            "first": s.first_valid_index().date(),
            "last": s.last_valid_index().date(),
            "observations": len(s),
            "missing": int(s.isna().sum()),
        }
        for sid, s in raw.items()
    }
).T

fintechco_demo.fred_client | VIXCLS: using cache VIXCLS_2026-09-25.csv


fintechco_demo.fred_client | NASDAQCOM: using cache NASDAQCOM_2026-09-25.csv


fintechco_demo.fred_client | DFEDTAR: using cache DFEDTAR_2026-09-25.csv


fintechco_demo.fred_client | DFEDTARU: using cache DFEDTARU_2026-09-25.csv


fintechco_demo.fred_client | DGS2: using cache DGS2_2026-09-25.csv


,first,last,observations,missing
VIXCLS,1990-01-02,2026-09-22,9581,302
NASDAQCOM,1990-01-02,2026-09-24,9583,332
DFEDTAR,1990-01-01,2008-12-15,6924,0
DFEDTARU,2008-12-16,2026-09-25,6493,0
DGS2,1990-01-02,2026-09-24,9584,395


## 2 · Cleaning and feature construction

Three decisions matter here and are made explicit:

1. **One policy-rate series.** FRED retired the single target in Dec 2008 when the FOMC moved to a range. We splice `DFEDTAR` with the range's upper bound, so a move to the 0–25 bp range counts as a −75 bp cut.
2. **One trading calendar.** Everything is aligned to VIX trading days. Holiday mismatches between the bond and equity markets are forward-filled by **at most one day**, and the log shows how many values were filled.
3. **Event sample starts Feb 1994.** Before that, the FOMC did not announce its decisions, so markets had to infer them and the decision "day" is ambiguous.

In [3]:
target = cl.splice_policy_target(raw["DFEDTAR"], raw["DFEDTARU"])

daily = cl.align_daily(
    {"VIXCLS": raw["VIXCLS"], "DGS2": raw["DGS2"], "NASDAQCOM": raw["NASDAQCOM"]}, base="VIXCLS"
).rename(columns={"VIXCLS": "vix_index", "DGS2": "ust2y_pct", "NASDAQCOM": "nasdaq_index"})
daily["fed_target_pct"] = target.reindex(daily.index, method="ffill")
daily["realized_vol_pct"] = an.realized_vol(daily["nasdaq_index"])

# Policy events, mapped to the first trading day on/after each decision
events = cl.events_to_trading_days(cl.policy_events(target), daily.index)
vix, ust2y, pos = (
    daily["vix_index"].to_numpy(),
    daily["ust2y_pct"].to_numpy(),
    events["pos"].to_numpy(),
)
events["vix_pre"] = vix[pos - 1]  # VIX close the day before
events["dvix0"] = vix[pos] - vix[pos - 1]  # decision-day VIX change (points)
events["surprise_bp"] = (ust2y[pos] - ust2y[pos - 1]) * 100  # decision-day 2y yield change (bp)
events["crisis"] = events["vix_pre"] >= STRESS_VIX

ev = events.loc[EVENT_SAMPLE_START:].copy()
print(f"{len(daily):,} trading days, {daily.index.min():%Y-%m-%d} to {daily.index.max():%Y-%m-%d}")
print(
    f"{len(events)} policy moves since 1990; {len(ev)} since {EVENT_SAMPLE_START} used for event analysis"
)
ev.tail(8)

fintechco_demo.cleaning | VIXCLS: 0 values forward-filled, 0 still missing


fintechco_demo.cleaning | DGS2: 108 values forward-filled, 0 still missing


fintechco_demo.cleaning | NASDAQCOM: 34 values forward-filled, 0 still missing


9,279 trading days, 1990-01-02 to 2026-09-22
110 policy moves since 1990; 92 since 1994-02-04 used for event analysis


,change_bp,direction,target_after_pct,pos,trading_day,vix_pre,dvix0,surprise_bp,crisis
date,,,,,,,,,
2023-07-27,25.00,hike,5.50,8464,2023-07-27,13.19,1.22,9.00,False
2024-09-19,-50.00,cut,5.00,8761,2024-09-19,18.23,-1.90,-2.00,False
2024-11-08,-25.00,cut,4.75,8797,2024-11-08,15.20,-0.26,5.00,False
2024-12-19,-25.00,cut,4.50,8826,2024-12-19,27.62,-3.53,-3.00,True
2025-09-18,-25.00,cut,4.25,9018,2025-09-18,15.72,-0.02,5.00,False
2025-10-30,-25.00,cut,4.00,9048,2025-10-30,16.92,-0.01,2.00,False
2025-12-11,-25.00,cut,3.75,9078,2025-12-11,15.77,-0.92,-2.00,False
2026-09-17,25.00,hike,4.00,9275,2026-09-17,17.71,-2.27,-7.00,False


## 3 · Context: the Fed and VIX since 1990

In [4]:
charts.policy_and_vix(target.loc["1990":], daily["vix_index"], events).show()

In [5]:
# Who moves first? The Fed cuts into stress and hikes into calm.
ev.groupby("direction").agg(
    moves=("change_bp", "size"),
    avg_size_bp=("change_bp", "mean"),
    vix_day_before=("vix_pre", "mean"),
    share_in_stress=("crisis", "mean"),
    avg_2y_surprise_bp=("surprise_bp", "mean"),
)

,moves,avg_size_bp,vix_day_before,share_in_stress,avg_2y_surprise_bp
direction,,,,,
cut,40,-40.00,27.24,0.53,-4.58
hike,52,32.69,17.14,0.12,0.87


**Read this before any causal claim.** The average VIX before a cut is far higher than before a hike. Rate cuts are a *response* to volatility, so a naive comparison of "VIX after cuts" against "VIX after hikes" mostly measures why the Fed moved, not what the move did. Sections 4 and 5 are designed around this problem.

## 4 · Event study: do rate decisions move VIX?

For each decision we track the change in VIX from the close the day **before** the decision (day −1) through 20 trading days after it. Shaded bands are 95% confidence intervals across events.

In [6]:
paths, summaries = {}, {}
for direction in ["hike", "cut"]:
    g = ev[ev["direction"] == direction]
    paths[direction] = an.event_paths(daily["vix_index"], g["pos"], pre=5, post=20)
    summaries[direction] = an.summarize_paths(paths[direction])

charts.event_paths(summaries, "VIX change vs. day −1 (points)", "VIXCLS, DFEDTAR, DFEDTARU").show()

### Placebo test: are decision days special?

VIX is **mean-reverting**. It tends to fall after it has been high, which is exactly when the Fed cuts. To avoid giving the Fed credit for ordinary mean reversion, we compare each group against **2,000 random samples of non-decision days, matched to the same VIX decile** on day −1. The empirical p-value is the share of placebo samples at least as extreme as the real decisions.

In [7]:
rows, null_h5 = [], {}
for direction in ["hike", "cut"]:
    event_pos = ev.loc[ev["direction"] == direction, "pos"]
    for h in [1, 5, 20]:
        null = an.placebo_mean_changes(daily["vix_index"], event_pos, horizon=h)
        observed = paths[direction][h].mean()
        rows.append(
            {
                "direction": direction,
                "horizon_days": h,
                "actual_mean_change": observed,
                "placebo_mean_change": null.mean(),
                "difference": observed - null.mean(),
                "placebo_95_low": np.percentile(null, 2.5),
                "placebo_95_high": np.percentile(null, 97.5),
                "p_value": an.empirical_p_value(observed, null),
            }
        )
        if h == 5:
            null_h5[direction] = null

placebo = pd.DataFrame(rows).set_index(["direction", "horizon_days"])
placebo

actual_mean_change  placebo_mean_change  difference  \
direction horizon_days                                                        
hike      1                          -0.25                 0.12       -0.37   
          5                           0.11                 0.30       -0.19   
          20                         -0.06                 0.58       -0.64   
cut       1                          -0.02                -0.34        0.32   
          5                          -0.39                -0.85        0.47   
          20                         -1.38                -1.75        0.37   

                        placebo_95_low  placebo_95_high  p_value  
direction horizon_days                                            
hike      1                      -0.34             0.65     0.14  
          5                      -0.46             1.16     0.64  
          20                     -0.57             1.97     0.29  
cut       1                      -1.18             0.56     0.46  
          5                      -2.01             0.27     0.42  
          20                     -3.19            -0.25     0.64

In [8]:
charts.placebo_histogram(
    pd.DataFrame(null_h5), {d: paths[d][5].mean() for d in ["hike", "cut"]}, horizon=5
).show()

## 5 · Policy regimes: is volatility different when the Fed is tightening vs. easing?

Each day is labeled by the change in the policy target over the prior ~6 months (126 trading days). We check both implied volatility (VIX) and **realized** 21-day NASDAQ volatility, so the result doesn't depend on a single measure.

In [9]:
daily["regime"] = an.policy_regime(daily["fed_target_pct"], lookback=126)
reg = daily.dropna(subset=["regime", "realized_vol_pct"])

regime_table = (
    reg.groupby("regime")
    .agg(
        days=("vix_index", "size"),
        vix_mean=("vix_index", "mean"),
        vix_median=("vix_index", "median"),
        vix_p90=("vix_index", lambda s: s.quantile(0.9)),
        realized_vol_mean=("realized_vol_pct", "mean"),
    )
    .loc[["tightening", "on hold", "easing"]]
)
regime_table

,days,vix_mean,vix_median,vix_p90,realized_vol_mean
regime,,,,,
tightening,2718,16.55,14.77,24.36,18.38
on hold,3642,18.94,17.16,27.80,17.76
easing,2793,22.81,20.29,33.09,24.58


In [10]:
charts.regime_boxes(reg, "vix_index", "VIX", "VIXCLS, DFEDTAR, DFEDTARU").show()

## 6 · Quantifying the effect: policy *surprises*

Well-telegraphed moves are priced in before they happen, so the size of the move is a weak signal. What matters is the **surprise**. We proxy it with the 2-year Treasury yield change on the decision day: positive means more hawkish than expected.

**Regression:** `ΔVIX(decision day) = α + β · surprise_bp`, OLS with heteroskedasticity-robust (HC1) standard errors. The effect is scaled to a **+25 bp hawkish surprise**. As a benchmark we run the same regression on *every* trading day, which checks whether decision days differ from normal risk-on/risk-off co-movement between yields and VIX.

In [11]:
def surprise_fit(frame: pd.DataFrame, label: str) -> dict:
    fit = smf.ols("dvix0 ~ surprise_bp", data=frame).fit(cov_type="HC1")
    b, se = fit.params["surprise_bp"], fit.bse["surprise_bp"]
    return {
        "sample": label,
        "n": int(fit.nobs),
        "vix_pts_per_25bp": 25 * b,
        "ci_low": 25 * (b - 1.96 * se),
        "ci_high": 25 * (b + 1.96 * se),
        "p_value": fit.pvalues["surprise_bp"],
        "r_squared": fit.rsquared,
        "_line": (fit.params["Intercept"], b),
    }


all_days = pd.DataFrame(
    {"dvix0": daily["vix_index"].diff(), "surprise_bp": daily["ust2y_pct"].diff() * 100}
).dropna()

fits = [
    surprise_fit(ev, "All decisions (1994+)"),
    surprise_fit(ev[~ev["crisis"]], f"Routine decisions (VIX < {STRESS_VIX})"),
    surprise_fit(ev[ev["crisis"]], f"Stress-period decisions (VIX ≥ {STRESS_VIX})"),
    surprise_fit(all_days, "Benchmark: every trading day"),
]
surprise_table = pd.DataFrame(fits).drop(columns="_line").set_index("sample")
surprise_table

,n,vix_pts_per_25bp,ci_low,ci_high,p_value,r_squared
sample,,,,,,
All decisions (1994+),92,-2.17,-4.24,-0.09,0.04,0.07
Routine decisions (VIX < 25),65,0.06,-1.16,1.27,0.92,0.00
Stress-period decisions (VIX ≥ 25),27,-3.97,-6.57,-1.36,0.00,0.15
Benchmark: every trading day,9278,-1.17,-1.40,-0.93,0.00,0.02


In [12]:
charts.surprise_scatter(
    ev, {"Fit: all decisions": fits[0]["_line"], "Fit: routine decisions": fits[1]["_line"]}
).show()

### How long does a surprise matter? (local projections)

We re-estimate the regression with the VIX change measured from day −1 to day *h*, for *h* = 0…20. A lasting effect would keep the line away from zero.

In [13]:
lp_all = an.surprise_response(daily["vix_index"], ev, range(0, 21))
lp_routine = an.surprise_response(daily["vix_index"], ev[~ev["crisis"]], range(0, 21))
charts.surprise_response(lp_all, lp_routine).show()
lp_all.loc[[0, 1, 5, 10, 20]].join(lp_routine.loc[[0, 1, 5, 10, 20]], rsuffix="_routine")

,effect,ci_low,ci_high,p_value,n,effect_routine,ci_low_routine,ci_high_routine,p_value_routine,n_routine
horizon,,,,,,,,,,
0,-2.17,-4.24,-0.09,0.04,92,0.06,-1.16,1.27,0.92,65
1,-0.47,-2.66,1.71,0.67,92,0.76,-0.36,1.87,0.18,65
5,0.37,-2.19,2.92,0.78,91,0.74,-0.77,2.25,0.34,64
10,0.65,-1.58,2.88,0.57,91,0.88,-1.03,2.79,0.37,64
20,0.04,-2.30,2.38,0.97,91,0.27,-1.96,2.50,0.81,64


## 7 · Caveats

- **Endogeneity.** The Fed reacts to markets. Regime averages (Section 5) describe co-movement, not causation.
- **Surprise proxy.** The 2-year yield's daily close also absorbs other same-day news, such as data releases. Fed funds futures or intraday windows would isolate the policy surprise more cleanly, but those data are licensed and not on FRED.
- **Small samples.** There are about 90 decisions since 1994, and the stress-period subset is small. Individual events like the reopening after 9/11 (2001-09-17) or March 2020 carry a lot of weight. Hover over the scatter plot to see them.
- **No decision calendar.** FRED doesn't flag scheduled vs. intermeeting (emergency) moves, so "stress period" (VIX ≥ 25) is a transparent stand-in for that split.
- **Descriptive, not a trading signal.** Nothing here has been tested out of sample.

## 8 · Next steps for the team

1. **Add "hold" decisions.** Most FOMC meetings end with no change, and a hawkish hold can be a surprise too. Adding an FOMC meeting calendar would capture those.
2. **Better surprise measures:** Fed funds futures (Kuttner-style), or the Bauer–Swanson / Nakamura–Steinsson shock series.
3. **More volatility measures:** rates volatility (MOVE), FX volatility, credit spreads (`BAMLH0A0HYM2`), which is more relevant to lending and fraud-loss models.
4. **Portfolio translation:** map the regime results onto hedging costs or VaR limits for the client-presentation use case.

### Extending this with Claude Code

Every module in `src/fintechco_demo/` is small and tested, so extensions are one prompt away. For example:

- *"Add the MOVE index and high-yield spreads to the regime table and tell me if the tightening/easing gap holds for rates volatility."*
- *"Split the surprise regression into hawkish vs. dovish surprises and test whether the response is asymmetric."*
- *"Turn Sections 3 and 6 into a Streamlit page in `app/dashboard.py` with a date-range filter."*

## 9 · Conclusion

**Question:** *How do Federal Reserve policy changes historically affect market volatility, and can we quantify the relationship?*

**Answer:** Much less than usually assumed. The dominant relationship runs the other way: **market volatility drives Fed policy.** Once that is accounted for, routine rate decisions have no measurable effect on VIX. The effect we *can* quantify is narrow: **surprise** decisions made **during market stress**, and it lasts about one day.

| What we tested | Result | Confidence |
|---|---|---|
| Does the Fed respond to volatility? | Yes. VIX averages 27.2 before cuts vs. 17.1 before hikes, and 53% of cuts come with VIX ≥ 25. | High |
| Do rate decisions move VIX (1–20 days)? | No. Hikes and cuts are indistinguishable from random days at the same VIX level (all p ≥ 0.14). | High |
| Is volatility different across policy regimes? | Yes: mean VIX 22.8 when easing vs. 16.6 when tightening. But this reflects *why* the Fed eases, not an effect of easing. | High (as correlation) |
| Do policy **surprises** move VIX? | Routine decisions: no (≈ 0 VIX pts per 25 bp, p = 0.92). Stress periods: yes, **−4.0 VIX pts per +25 bp hawkish surprise** (p < 0.01), gone by the next day. | Moderate (n = 27 stress events) |

### What this means for FinTechCo

1. **Don't treat the FOMC calendar as a volatility event.** Hedging or de-risking ahead of routine, well-communicated decisions has historically bought little protection.
2. **Watch market stress and surprises instead.** The risk case is an unexpectedly *dovish* Fed during turbulence. Markets read it as the Fed seeing more trouble than expected, and implied volatility jumps that day.
3. **Treat regime labels as context, not a forecast.** "The Fed is easing" tells you volatility is probably already elevated. It doesn't tell you volatility is about to fall.

### Bottom line

For client conversations: *"Fed decisions themselves rarely move markets. Volatility comes from the conditions that force the Fed's hand, and from surprises in a crisis."* Before building this into portfolio or risk models, validate it with a cleaner surprise measure (Fed funds futures) and out-of-sample testing (see Section 8).